# Création du CLIP

In [ ]:
import os
import pandas as pd
import re
import numpy as np
import random
import zipfile
import requests
import io
import math
from pathlib import Path
import cv2

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.utils import register_keras_serializable, to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.models import load_model
from tensorflow.keras.metrics import Mean
from tensorflow.keras.layers import Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import load_model, load_weights

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Pour utiliser au mieux le GPU
AUTOTUNE = tf.data.AUTOTUNE

## Récupération des architectures et poids des modèles de classification de texte et d'image

In [ ]:
saved_models_path = "./src/models_forclip"

model_images = load_model.(os.path.join(saved_models_path, "best_image_classif.keras"))
model_text = load_model.(os.path.join(saved_models_path, "best_smallbert.keras"))

print("Modèle Image")
model_images.summary()

print("Modèle Texte")
model_text.summary()

### Élagage et reconstruction des 2 modèles

In [ ]:
learning_rate=1e-3
loss_images="categorical_crossentropy"
loss_text="sparse_categorical_crossentropy"
metrics=["accuracy"]

cropped_model_images = model_images.layers[-3].output
new_model_images = Model(inputs=model_images.input, outputs=cropped_model_images)
new_model_images.compile(optimizer=Adam(learning_rate), loss=loss_images, metrics=metrics)

cropped_model_text = model_text.layers[-1].output
new_model_text = Model(inputs=model_text.input, outputs=cropped_model_text)
new_model_text.compile(optimizer="adam", loss=loss_text, metrics=metrics)